# 11 - Screen known QAC determinants and explain errors

This notebook bundles the interpretable baseline: retrieve a curated public reference panel, screen the 197 existing assemblies with BLAST+, evaluate a known-determinant rule, and inspect tolerant isolates not explained by that rule. It does not download additional genomes.


## 1. Configuration

The reference panel contains `bcrABC`, `qacH`, `emrE`, and `emrC`. The default rule requires at least 80% nucleotide identity and 80% reference coverage. This is a compact hypothesis-oriented screen, not a replacement for a full resistome workflow.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
print("Project root:", PROJECT_ROOT)

MANIFEST = PROJECT_ROOT / "data" / "manifests" / "isolate_manifest_public_assemblies.csv"
REFERENCES = PROJECT_ROOT / "data" / "references" / "qac_determinants.fasta"
SCREEN = PROJECT_ROOT / "data" / "processed" / "qac_determinant_screen.csv"
REPORT = PROJECT_ROOT / "reports" / "qac_gene_rule_baseline.json"
RUN_REFERENCE_FETCH = False
RUN_QAC_SCREEN = False


## 2. Retrieve the small curated reference panel

Set `RUN_REFERENCE_FETCH = True` for the first run. This downloads only four short public NCBI sequence records.


In [ ]:
command = [sys.executable, str(PROJECT_ROOT / "scripts" / "fetch_qac_references.py"), "--output", str(REFERENCES)]
print(" ".join(command))
if RUN_REFERENCE_FETCH:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print("Reference FASTA exists:", REFERENCES.exists())


## 3. Run BLAST screening and evaluate the rule baseline

Set `RUN_QAC_SCREEN = True`. The screen reuses the compact FASTA assemblies already on disk.


In [ ]:
command = [
    sys.executable, str(PROJECT_ROOT / "scripts" / "screen_qac_determinants.py"),
    "--manifest", str(MANIFEST), "--references", str(REFERENCES),
    "--output", str(SCREEN), "--report", str(REPORT),
]
print(" ".join(command))
if RUN_QAC_SCREEN:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
report = json.loads(REPORT.read_text())
display(pd.Series(report))


## 4. Review determinant frequencies and unexplained tolerant isolates

The most interesting subset is tolerant isolates without one of the four screened determinants. Those isolates test whether ML contributes information beyond a simple known-gene rule.


In [ ]:
hits = pd.read_csv(SCREEN)
manifest = pd.read_csv(MANIFEST)
presence = hits.pivot(index="isolate_id", columns="gene", values="present").reset_index()
review = manifest.merge(presence, on="isolate_id", validate="one_to_one")
gene_columns = ["bcrABC", "qacH", "emrE", "emrC"]
review["known_qac_gene"] = review[gene_columns].any(axis=1)
print("Gene-positive counts:")
display(review[gene_columns].sum().sort_values(ascending=False).to_frame("isolates"))
print("Tolerance by known-QAC rule:")
display(pd.crosstab(review["known_qac_gene"], review["label"], margins=True))
print("Tolerant isolates not explained by the four-reference panel:")
display(review[(review["label"] == 1) & ~review["known_qac_gene"]][["isolate_id", "group", "mic_bc_mg_l", "source_metadata"]])


## 5. Decision

Notebook 12 is worth running if DNABERT2 should be tested as a complement to the known-gene rule. Keep the claim narrow: the rule screen is interpretable but intentionally compact, while a manuscript workflow should later add a full annotated resistome pipeline.
